# LeetCode 3699 — Number of ZigZag Arrays I bằng Python

- Link: https://leetcode.com/problems/number-of-zigzag-arrays-i/
- Độ khó: **Hard**
- Chủ đề: DP đếm dãy theo **giá trị cuối và hướng bước cuối**, **tổng tiền tố/hậu tố**, khai thác **đối xứng** `v ↔ m−1−v`

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def zigZagArrays(self, n: int, l: int, r: int) -> int:
```

## 1. Mục tiêu học

1. Đặt trạng thái DP `(giá trị cuối, bước cuối đi lên hay đi xuống)`.
2. Chuyển trạng thái bằng **tổng tiền tố**: lên tới `v` cần bước trước là xuống và giá trị trước `< v`.
3. Giảm từ `O(n·m²)` xuống `O(n·m)` (`m = r − l + 1`).
4. Dùng **đối xứng phản chiếu** để chỉ giữ một mảng: `D[v] = U[m−1−v]`.
5. Kiểm chứng bằng vét cạn mọi dãy.

## 2. Đề bài & ví dụ

Dãy ZigZag độ dài `n`: mọi phần tử thuộc `[l, r]`; hai phần tử kề nhau khác nhau; **không** có ba phần tử liên tiếp tăng ngặt hoặc giảm ngặt. Đếm số dãy, modulo `10^9 + 7`.

**Ví dụ 1:** `n = 3, l = 4, r = 5` → `2` (`[4,5,4]`, `[5,4,5]`).

**Ví dụ 2:** `n = 3, l = 1, r = 3` → `10`.

**Ràng buộc:** `3 ≤ n ≤ 2000`, `1 ≤ l < r ≤ 2000`.

In [1]:
from itertools import product, accumulate
import random, time

MOD = 10**9 + 7
examples = [(3, 4, 5, 2), (3, 1, 3, 10)]
for n, l, r, exp in examples:
    print(n, l, r, "-> kỳ vọng", exp)

3 4 5 -> kỳ vọng 2
3 1 3 -> kỳ vọng 10


## 3. Kiến thức nền

### 3.1. ZigZag = hướng bước luân phiên

Hai phần tử kề nhau khác nhau ⇒ mỗi bước là "lên" hoặc "xuống". Không có ba phần tử tăng/giảm liên tiếp ⇔ hai bước liên tiếp **khác hướng** ⇒ dãy hướng là `lên, xuống, lên, …` hoặc `xuống, lên, xuống, …`. Chỉ giá trị (không phải `l`, `r` cụ thể) quan trọng ⇒ quy về `0..m−1`.

In [2]:
def is_zigzag(a):
    if any(a[i] == a[i + 1] for i in range(len(a) - 1)): return False
    return not any((a[i] < a[i + 1] < a[i + 2]) or (a[i] > a[i + 1] > a[i + 2]) for i in range(len(a) - 2))

print(is_zigzag([1, 3, 2, 3]), is_zigzag([1, 2, 3]), is_zigzag([2, 2, 1]))

True False False


### 3.2. Tổng tiền tố để chuyển trạng thái

`Σ_{u < v} X[u]` cho mọi `v` = tiền tố của `X` ⇒ tính cả mảng trong `O(m)` với `itertools.accumulate`.

In [3]:
X = [3, 1, 4, 1, 5]
pre = [0] + list(accumulate(X))
print("Σ_{u<v} X[u] =", pre[:-1])

Σ_{u<v} X[u] = [0, 3, 4, 8, 9]


### 3.3. Đối xứng phản chiếu

Ánh xạ `v ↦ m − 1 − v` biến dãy ZigZag thành dãy ZigZag, đổi mọi bước "lên" thành "xuống". Vậy số dãy kết thúc tại `v` bằng bước lên = số dãy kết thúc tại `m−1−v` bằng bước xuống.

### 3.4. Khởi tạo từ độ dài 1 cũng được

Nếu coi dãy một phần tử "vừa đi lên vừa đi xuống" (`U_1[v] = D_1[v] = 1`), một bước chuyển trạng thái cho đúng `U_2[v] = v`, `D_2[v] = m − 1 − v`. Hai cách khởi tạo tương đương — điều quan trọng là **không** đếm dãy độ dài 2 hai lần.

In [4]:
m = 5
U1 = [1] * m; D1 = [1] * m
U2 = [sum(D1[:v]) for v in range(m)]; D2 = [sum(U1[v + 1:]) for v in range(m)]
print("U_2 =", U2, "| D_2 =", D2, "| = [v] và [m-1-v]:", U2 == list(range(m)) and D2 == [m - 1 - v for v in range(m)])

U_2 = [0, 1, 2, 3, 4] | D_2 = [4, 3, 2, 1, 0] | = [v] và [m-1-v]: True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

`U_i[v]` = số dãy độ dài `i` kết thúc tại `v` với bước cuối **lên**; `D_i[v]` tương tự với bước **xuống**.

- `U_{i+1}[v] = Σ_{u < v} D_i[u]` (bước trước phải là xuống).
- `D_{i+1}[v] = Σ_{u > v} U_i[u]`.
- Khởi tạo `i = 2`: `U_2[v] = v` (số `u < v`), `D_2[v] = m − 1 − v`.
- Đáp án `Σ_v (U_n[v] + D_n[v])`.

Ví dụ 2 (`m = 3`): `U_2 = [0,1,2]`, `D_2 = [2,1,0]`; `U_3 = [0, 2, 3]`, `D_3 = [3, 2, 0]` ⇒ tổng `10`. ✔

## 5. Cách 1 — Brute Force

In [5]:
def brute(n, l, r):
    return sum(1 for a in product(range(l, r + 1), repeat=n) if is_zigzag(a)) % MOD

for n, l, r, exp in examples:
    print(brute(n, l, r), "kỳ vọng", exp)

2 kỳ vọng 2
10 kỳ vọng 10


**Nút thắt:** `m^n` dãy.

## 6. Ý tưởng thuật toán / bước đột phá

DP mục 4 với tổng tiền tố cho `O(n·m)`. Thêm đối xứng: `D_i[v] = U_i[m−1−v]` ⇒

`U_{i+1}[v] = Σ_{u<v} U_i[m−1−u] = Σ_{w ≥ m−v} U_i[w]` — một **tổng hậu tố** của chính `U_i`.

Đáp án `= 2 · Σ_v U_n[v]`. Chỉ một mảng, một tổng hậu tố mỗi bước.

## 7. Các cách tiếp theo

- **Cách 2:** DP `U/D` không tổng tiền tố — `O(n·m²)`.
- **Cách 3:** DP `U/D` với tổng tiền tố — `O(n·m)`.
- **Cách 4:** DP một mảng nhờ đối xứng — `O(n·m)`, hằng số nhỏ hơn (lời giải nộp).

### 7.1. Cách 2 — DP `O(n·m²)`

In [6]:
def dp_quadratic(n, l, r):
    m = r - l + 1
    U = list(range(m)); D = [m - 1 - v for v in range(m)]
    for _ in range(n - 2):
        U, D = ([sum(D[u] for u in range(v)) % MOD for v in range(m)],
                [sum(U[u] for u in range(v + 1, m)) % MOD for v in range(m)])
    return (sum(U) + sum(D)) % MOD

for n, l, r, exp in examples:
    print(dp_quadratic(n, l, r), "kỳ vọng", exp)

2 kỳ vọng 2
10 kỳ vọng 10


### 7.2. Cách 3 — DP với tổng tiền tố

In [7]:
def dp_prefix(n, l, r):
    m = r - l + 1
    U = list(range(m)); D = [m - 1 - v for v in range(m)]
    for _ in range(n - 2):
        preD = [0] + list(accumulate(D))                       # preD[v] = Σ_{u<v} D[u]
        sufU = list(accumulate(reversed(U)))[::-1] + [0]      # sufU[v] = Σ_{u>=v} U[u]
        U, D = [preD[v] % MOD for v in range(m)], [sufU[v + 1] % MOD for v in range(m)]
    return (sum(U) + sum(D)) % MOD

for n, l, r, exp in examples:
    print(dp_prefix(n, l, r), "kỳ vọng", exp)

2 kỳ vọng 2
10 kỳ vọng 10


### 7.3. Cách 4 — Một mảng nhờ đối xứng (lời giải chính)

In [8]:
def dp_symmetric(n, l, r):
    m = r - l + 1
    U = list(range(m))
    for _ in range(n - 2):
        suf = list(accumulate(reversed(U)))[::-1] + [0]        # suf[w] = Σ_{x >= w} U[x]
        U = [suf[m - v] % MOD for v in range(m)]
    return 2 * sum(U) % MOD

for n, l, r, exp in examples:
    print(dp_symmetric(n, l, r), "kỳ vọng", exp)

2 kỳ vọng 2
10 kỳ vọng 10


### 7.4. Góc nhìn ma trận (cầu nối sang bản II)

Phép `U ↦ U'` với `U'[v] = Σ_{w ≥ m−v} U[w]` là **tuyến tính**: `U' = T·U`, với `T[v][w] = 1` khi `w ≥ m − v`. Ma trận `T` có dạng "tam giác dưới lật": hàng `v` có đúng `v` số 1 ở cuối. Vì vậy `U_n = T^{n−2} U_2`. Bản I (`n ≤ 2000`) nhân trực tiếp là đủ; bản II (`n ≤ 10^9`, `m ≤ 75`) dùng **luỹ thừa nhanh** của `T`.

In [9]:
def transition_matrix(m):
    return [[1 if w >= m - v else 0 for w in range(m)] for v in range(m)]

m = 4; T = transition_matrix(m)
for row in T: print(row)
U = list(range(m))
for _ in range(3):
    U_mat = [sum(T[v][w] * U[w] for w in range(m)) for v in range(m)]
    suf = list(accumulate(reversed(U)))[::-1] + [0]
    U_dp = [suf[m - v] for v in range(m)]
    assert U_mat == U_dp; U = U_dp
print("T·U khớp công thức hậu tố qua 3 bước; U_5 =", U)

[0, 0, 0, 0]
[0, 0, 0, 1]
[0, 0, 1, 1]
[0, 1, 1, 1]
T·U khớp công thức hậu tố qua 3 bước; U_5 = [0, 14, 25, 31]


### 7.5. Trường hợp `m = 2`

Chỉ có hai giá trị ⇒ dãy phải luân phiên tuyệt đối `a, b, a, b, …` ⇒ đúng **2** dãy với mọi `n`. Đây là ca kiểm tra nhanh cho mọi cài đặt (xem 12.3).

## 8. TRACE từng bước

In [10]:
def trace(n, l, r):
    m = r - l + 1
    U = list(range(m)); D = [m - 1 - v for v in range(m)]
    print(f"m = {m}; i = 2: U = {U}, D = {D}, tổng = {sum(U) + sum(D)}")
    for i in range(3, n + 1):
        U, D = ([sum(D[:v]) for v in range(m)], [sum(U[v + 1:]) for v in range(m)])
        print(f"i = {i}: U = {U}, D = {D}, tổng = {sum(U) + sum(D)} | D đảo ngược = {D[::-1]} (= U?)")

trace(5, 1, 4)
print("brute n=5, [1,4]:", brute(5, 1, 4))

m = 4; i = 2: U = [0, 1, 2, 3], D = [3, 2, 1, 0], tổng = 12
i = 3: U = [0, 3, 5, 6], D = [6, 5, 3, 0], tổng = 28 | D đảo ngược = [0, 3, 5, 6] (= U?)
i = 4: U = [0, 6, 11, 14], D = [14, 11, 6, 0], tổng = 62 | D đảo ngược = [0, 6, 11, 14] (= U?)
i = 5: U = [0, 14, 25, 31], D = [31, 25, 14, 0], tổng = 140 | D đảo ngược = [0, 14, 25, 31] (= U?)
brute n=5, [1,4]: 140


In [11]:
# liệt kê các dãy của ví dụ 2 theo hướng bước cuối
seqs = [a for a in product(range(1, 4), repeat=3) if is_zigzag(a)]
print("kết thúc bằng bước lên:", [a for a in seqs if a[-2] < a[-1]])
print("kết thúc bằng bước xuống:", [a for a in seqs if a[-2] > a[-1]])

kết thúc bằng bước lên: [(2, 1, 2), (2, 1, 3), (3, 1, 2), (3, 1, 3), (3, 2, 3)]
kết thúc bằng bước xuống: [(1, 2, 1), (1, 3, 1), (1, 3, 2), (2, 3, 1), (2, 3, 2)]


## 9. Code nộp LeetCode

In [12]:
from itertools import accumulate

class Solution:
    def zigZagArrays(self, n: int, l: int, r: int) -> int:
        MOD = 10**9 + 7
        m = r - l + 1
        U = list(range(m))                       # độ dài 2: U[v] = số u < v
        for _ in range(n - 2):
            suf = list(accumulate(reversed(U)))[::-1]
            suf.append(0)
            U = [suf[m - v] % MOD for v in range(m)]
        return 2 * sum(U) % MOD

sol = Solution()
for n, l, r, exp in examples:
    got = sol.zigZagArrays(n, l, r)
    print(n, l, r, "->", got, "✔" if got == exp else "✘")

3 4 5 -> 2 ✔
3 1 3 -> 10 ✔


## 10. Vì sao thuật toán đúng

1. **Đặc trưng:** dãy ZigZag ⇔ mọi bước khác 0 và hai bước liên tiếp ngược dấu (3.1).
2. **DP `U/D`:** dãy độ dài `i+1` kết thúc tại `v` bằng bước lên ⇔ phần đầu độ dài `i` kết thúc tại `u < v` và (nếu `i ≥ 2`) bước cuối của nó là xuống ⇒ `U_{i+1}[v] = Σ_{u<v} D_i[u]`. Tương tự cho `D`. Với `i = 2`, mọi cặp `u < v` đều hợp lệ ⇒ `U_2[v] = v`.
3. **Đối xứng:** phép phản chiếu là song ánh trên tập dãy ZigZag, đổi hướng mọi bước ⇒ `D_i[v] = U_i[m−1−v]` với mọi `i` (quy nạp, hoặc trực tiếp từ song ánh).
4. **Thay vào:** `U_{i+1}[v] = Σ_{u=0}^{v−1} U_i[m−1−u] = Σ_{w=m−v}^{m−1} U_i[w]`; đáp án `Σ U + Σ D = 2 Σ U`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(m^n · n)` | `O(n)` |
| 2. DP `O(n m²)` | `O(n m²)` | `O(m)` |
| 3. DP + tiền tố | `O(n m)` | `O(m)` |
| 4. DP đối xứng | `O(n m)` | `O(m)` |

Với `n = m = 2000`: `4·10^6` phép cộng — chạy bằng `accumulate` (C) nên nhanh.

## 12. Bộ test

In [13]:
METHODS = [brute, dp_quadratic, dp_prefix, dp_symmetric, lambda n, l, r: Solution().zigZagArrays(n, l, r)]
for n, l, r, exp in examples:
    assert all(f(n, l, r) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [14]:
# 12.2 lưới nhỏ đầy đủ so với vét cạn
for n in range(3, 7):
    for l in range(1, 4):
        for r in range(l + 1, 7):
            b = brute(n, l, r)
            assert all(f(n, l, r) == b for f in METHODS[1:]), (n, l, r)
print("Lưới n ≤ 6, m ≤ 6: OK")

Lưới n ≤ 6, m ≤ 6: OK


In [15]:
# 12.3 ngẫu nhiên cỡ vừa: các cách DP phải khớp
rng = random.Random(3699)
for _ in range(100):
    n = rng.randint(3, 60); l = rng.randint(1, 50); r = rng.randint(l + 1, 80)
    res = [f(n, l, r) for f in METHODS[1:]]
    assert len(set(res)) == 1, (n, l, r, res)
assert Solution().zigZagArrays(3, 1, 2) == 2 and Solution().zigZagArrays(10, 1, 2) == 2   # m = 2: chỉ 2 dãy luân phiên
print("100 test ngẫu nhiên: OK")

100 test ngẫu nhiên: OK


In [16]:
# 12.4 cận lớn nhất
t0 = time.time(); x = Solution().zigZagArrays(2000, 1, 2000); t1 = time.time()
print("n = m = 2000:", x, "| thời gian:", round(t1 - t0, 2), "s | khớp Cách 3:", x == dp_prefix(2000, 1, 2000))

n = m = 2000: 594850306 | thời gian: 0.4 s | khớp Cách 3: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ cấm "ba phần tử bằng nhau" hoặc quên cấm hai phần tử kề bằng nhau

In [17]:
def wrong_allow_equal(n, l, r):
    ok = lambda a: not any((a[i] < a[i + 1] < a[i + 2]) or (a[i] > a[i + 1] > a[i + 2]) for i in range(len(a) - 2))
    return sum(1 for a in product(range(l, r + 1), repeat=n) if ok(a))       # SAI: cho phép kề bằng nhau

print("sai:", wrong_allow_equal(3, 4, 5), "| đúng:", brute(3, 4, 5))

sai: 8 | đúng: 2


### Lỗi 2 — Dùng đối xứng nhưng quên nhân 2 ở cuối

Mảng `U` chỉ đếm các dãy kết thúc bằng bước **lên**; các dãy kết thúc bằng bước xuống có cùng số lượng (đối xứng) nên phải nhân đôi.

In [18]:
def wrong_no_double(n, l, r):
    m = r - l + 1; U = list(range(m))
    for _ in range(n - 2):
        suf = list(accumulate(reversed(U)))[::-1] + [0]; U = [suf[m - v] % MOD for v in range(m)]
    return sum(U) % MOD                                                          # SAI: thiếu ×2

print("sai:", wrong_no_double(3, 1, 3), "| đúng:", brute(3, 1, 3))

sai: 5 | đúng: 10


### Lỗi 3 — Đối xứng sai chỉ số (`m − v` thay vì `m − 1 − v`)

In [19]:
def wrong_index(n, l, r):
    m = r - l + 1; U = list(range(m))
    for _ in range(n - 2):
        suf = list(accumulate(reversed(U)))[::-1] + [0]
        U = [suf[min(m, m - v + 1)] % MOD for v in range(m)]                    # SAI: lệch 1
    return 2 * sum(U) % MOD

print("sai:", wrong_index(4, 1, 4), "| đúng:", brute(4, 1, 4))

sai: 26 | đúng: 62


### Lỗi 4 — Quên `% MOD` trong vòng lặp

Với `n = 2000`, số lượng tăng theo hàm mũ — số nguyên Python vẫn đúng nhưng **chậm dần** (số hàng nghìn chữ số). Lấy modulo mỗi bước.

In [20]:
m = 50; U = list(range(m))
for _ in range(300):
    suf = list(accumulate(reversed(U)))[::-1] + [0]; U = [suf[m - v] for v in range(m)]     # không modulo
print("số chữ số của U[-1] sau 300 bước (không modulo):", len(str(U[-1])))

số chữ số của U[-1] sau 300 bước (không modulo): 452


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Không ba phần tử đơn điệu liên tiếp" ⇒ hướng bước luân phiên ⇒ trạng thái `(giá trị cuối, hướng)`.
- Chuyển trạng thái dạng `Σ_{u<v}` ⇒ tổng tiền tố.
- Bài có đối xứng phản chiếu ⇒ giảm một nửa trạng thái.
- `n` tới `10^9` (bản II) ⇒ chuyển trạng thái tuyến tính ⇒ **luỹ thừa ma trận**.
- Trước khi cài, kiểm tra công thức trên các ca "suy biến" có đáp án biết trước: `m = 2` (luôn 2 dãy), `n = 3` (đếm tay được).
- Khi trạng thái có hai "nửa" đối xứng nhau, hãy viết bản đầy đủ trước (Cách 3) rồi mới rút gọn (Cách 4) — dễ kiểm chứng từng bước.

## 15. Biến thể của bài

1. **Bản II** (`n ≤ 10^9`, `m ≤ 75`): luỹ thừa ma trận `m × m` (nhờ đối xứng).
2. **Cho phép bằng nhau** nhưng cấm đơn điệu ngặt ba phần tử: thêm trạng thái "bước ngang".
3. **Bắt buộc bắt đầu bằng bước lên**: chỉ lấy một nửa (= đáp án / 2).
4. **Hoán vị ZigZag** (alternating permutations, số Euler zigzag): DP Entringer.

In [21]:
# Biến thể 4: số hoán vị zigzag (luân phiên) của 1..n — tam giác Entringer
def alternating_permutations(n):
    E = [[0] * (n + 1) for _ in range(n + 1)]; E[0][0] = 1
    for i in range(1, n + 1):
        for k in range(1, i + 1):
            E[i][k] = E[i][k - 1] + E[i - 1][i - k]
    return 2 * E[n][n] if n >= 2 else 1

from itertools import permutations
for n in range(2, 8):
    assert alternating_permutations(n) == sum(1 for p in permutations(range(n)) if is_zigzag(p))
print("Biến thể 4 OK:", [alternating_permutations(n) for n in range(2, 10)])

Biến thể 4 OK: [2, 4, 10, 32, 122, 544, 2770, 15872]


**Biến thể 2 (cho phép kề bằng nhau, chỉ cấm ba phần tử đơn điệu ngặt).** Bây giờ có ba loại bước: lên, xuống, ngang. Cấm "lên rồi lên" và "xuống rồi xuống"; bước ngang không tạo đơn điệu ngặt nên sau nó được đi mọi hướng. Trạng thái `(giá trị cuối, bước cuối ∈ {lên, xuống, ngang/bắt đầu})`.

In [22]:
def count_allow_equal(n, l, r):
    m = r - l + 1
    U = list(range(m)); D = [m - 1 - v for v in range(m)]; F = [1] * m     # F: bước cuối ngang
    for _ in range(n - 2):
        preDF = [0] + list(accumulate(D[v] + F[v] for v in range(m)))
        sufUF = list(accumulate(reversed([U[v] + F[v] for v in range(m)])))[::-1] + [0]
        U, D, F = ([preDF[v] % MOD for v in range(m)], [sufUF[v + 1] % MOD for v in range(m)],
                   [(U[v] + D[v] + F[v]) % MOD for v in range(m)])
    return (sum(U) + sum(D) + sum(F)) % MOD

def brute_allow_equal(n, l, r):
    ok = lambda a: not any((a[i] < a[i + 1] < a[i + 2]) or (a[i] > a[i + 1] > a[i + 2]) for i in range(len(a) - 2))
    return sum(1 for a in product(range(l, r + 1), repeat=n) if ok(a))

for n in range(3, 7):
    for r in range(2, 6):
        assert count_allow_equal(n, 1, r) == brute_allow_equal(n, 1, r), (n, r)
print("Biến thể 2 OK:", count_allow_equal(3, 4, 5), "dãy (so với 2 dãy ZigZag)")

Biến thể 2 OK: 8 dãy (so với 2 dãy ZigZag)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1. Vét cạn | Mọi dãy | `O(m^n)` | Kiểm chứng |
| 2. DP `m²` | Tổng trực tiếp | `O(n m²)` | `m` nhỏ |
| 3. DP + tiền tố | `U/D` | `O(n m)` | Rõ ràng |
| 4. DP đối xứng | Một mảng | `O(n m)` | **Lời giải chuẩn** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `zigZagArrays(4, 1, 2)`.

**Bài 2 (tính tay).** `zigZagArrays(4, 1, 3)` bằng bảng `U`.

**Bài 3 (điền TODO).** Viết `count_starting_up(n, l, r)` — chỉ đếm dãy có bước đầu tiên đi lên.

**Bài 4 (tự viết lại).** Viết lại lời giải không dùng `accumulate` (vòng lặp thường) và so tốc độ.

**Bài 5 (mini challenge).** Đếm dãy ZigZag độ dài `n` với **phần tử đầu tiên cố định** bằng `s`.

In [23]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def count_starting_up(n, l, r):
    # TODO
    pass

In [24]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def count_with_first(n, l, r, s):
    # TODO: khởi tạo U_2, D_2 chỉ từ u = s
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `m = 2` ⇒ chỉ có hai dãy luân phiên `1,2,1,2` và `2,1,2,1`. (2) Không còn lựa chọn khác. (3) `2`.

**Bài 2.** (1) `U_2 = [0,1,2]`, `U_3 = [0,2,3]`. (2) `U_4[v] = Σ_{w ≥ 3−v} U_3[w]` = `[0, 3, 5]`. (3) `2·8 = 16`.

**Bài 3.** (1) Theo đối xứng, nửa số dãy bắt đầu bằng bước lên. (2) Hoặc DP riêng. (3) Đáp án = tổng / 2.

**Bài 4.** (1) Tính `suf` bằng vòng lặp ngược. (2) Giữ công thức. (3) So kết quả và thời gian.

**Bài 5.** (1) `U_2[v] = [s < v]`, `D_2[v] = [s > v]`. (2) Không còn đối xứng ⇒ dùng `U/D` đầy đủ. (3) Chạy `n − 2` bước.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1, 2
assert Solution().zigZagArrays(4, 1, 2) == 2 == brute(4, 1, 2)
assert Solution().zigZagArrays(4, 1, 3) == 16 == brute(4, 1, 3)
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [26]:
# Bài 3, 4
def count_starting_up_sol(n, l, r):
    return Solution().zigZagArrays(n, l, r) * pow(2, MOD - 2, MOD) % MOD

def dp_loops(n, l, r):
    m = r - l + 1; U = list(range(m))
    for _ in range(n - 2):
        suf = [0] * (m + 1)
        for w in range(m - 1, -1, -1): suf[w] = (suf[w + 1] + U[w]) % MOD
        U = [suf[m - v] for v in range(m)]
    return 2 * sum(U) % MOD

for n, l, r in [(3, 1, 3), (5, 2, 6), (6, 1, 4)]:
    assert count_starting_up_sol(n, l, r) == sum(1 for a in product(range(l, r + 1), repeat=n) if is_zigzag(a) and a[0] < a[1])
t0 = time.time(); a1 = dp_loops(800, 1, 800); t1 = time.time(); a2 = Solution().zigZagArrays(800, 1, 800); t2 = time.time()
assert a1 == a2
print(f"Bài 3, 4 OK | vòng lặp thường {t1 - t0:.2f}s vs accumulate {t2 - t1:.2f}s")

Bài 3, 4 OK | vòng lặp thường 0.12s vs accumulate 0.05s


In [27]:
# Bài 5
def count_with_first_sol(n, l, r, s):
    m = r - l + 1; s -= l
    U = [1 if s < v else 0 for v in range(m)]; D = [1 if s > v else 0 for v in range(m)]
    for _ in range(n - 2):
        preD = [0] + list(accumulate(D)); sufU = list(accumulate(reversed(U)))[::-1] + [0]
        U, D = [preD[v] % MOD for v in range(m)], [sufU[v + 1] % MOD for v in range(m)]
    return (sum(U) + sum(D)) % MOD

for n, l, r in [(3, 1, 3), (4, 2, 5), (5, 1, 4)]:
    for s in range(l, r + 1):
        assert count_with_first_sol(n, l, r, s) == sum(1 for a in product(range(l, r + 1), repeat=n) if is_zigzag(a) and a[0] == s)
    assert sum(count_with_first_sol(n, l, r, s) for s in range(l, r + 1)) % MOD == Solution().zigZagArrays(n, l, r)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
m = r - l + 1
U = list(range(m))                     # độ dài 2, bước cuối lên
for _ in range(n - 2):
    suf = hậu tố của U (suf[m] = 0)
    U = [suf[m - v] for v in range(m)]   # U'[v] = Σ_{w >= m-v} U[w]
return 2 * sum(U) % MOD
```

**Checklist:**

- [ ] Cấm cả kề bằng nhau và ba phần tử đơn điệu ngặt?
- [ ] Khởi tạo ở độ dài 2 (`U_2[v] = v`)?
- [ ] Chỉ số đối xứng `m − 1 − v` (hậu tố từ `m − v`)?
- [ ] `% MOD` mỗi bước?
- [ ] Đã so với vét cạn trên lưới nhỏ?